# Demo: Access Control trên NoSQL Database với dữ liệu trường học
**Bám sát bài báo:** *Access Control on NoSQL Databases* — Vincent C. Hu, NIST IR 8504 (05/2024)

Notebook này **cài đặt lại bằng Python thuần** (mô phỏng NoSQL trong bộ nhớ, không cần cài MongoDB/Neo4j) các ý chính của bài báo, trên dataset thật về học sinh:

| Mục trong bài báo | Nội dung | Phần trong notebook |
|---|---|---|
| §2.1 | 4 mô hình NoSQL: key-value, wide-column, document, graph | Phần 1 |
| §3.1, Fig. 5–6 | Cấu trúc cây (tối đa 5 tầng) và đồ thị có hướng | Phần 2 |
| §3.2.1–3.2.2, Fig. 7–8 | Thuộc tính subject/object theo đường đi trong cây, policy tuple | Phần 3 |
| §3.2.3, Fig. 9–10 | Graph model: thuộc tính là chuỗi cạnh/nút, cạnh = hành động, RBAC bằng nút role | Phần 4 |
| §4.1–4.9 | Fine-grained AC, injection, query language, consistency, performance, audit, điều kiện môi trường, NLP→rule, kết hợp RDBMS | Phần 5 |
| Appendix A | Federation (RP/RM/RC), luật là tập quan hệ nhị phân, lưu dạng triple trong graph | Phần 6 |

**Về dữ liệu**
- Dữ liệu học sinh: **UCI "Student Performance"** (`student-mat.csv`, 395 học sinh, 33 cột, 2 trường `GP` và `MS`). Notebook tự tải từ UCI.
- Nếu bạn đã có file CSV, đặt đường dẫn vào `LOCAL_CSV` ở cell dưới. Khi chạy từ web **Sổ điểm**, web xuất dataset đang chọn từ MongoDB ra CSV và đặt biến này giúp bạn.
- Nếu không có mạng, notebook tự sinh dữ liệu tổng hợp **cùng schema** để vẫn chạy được (có in cảnh báo).
- Dataset **không có** thông tin người dùng hệ thống (giáo viên, tư vấn viên, quản trị). Phần này là **mô phỏng** để có subject cho access control.

In [ ]:
import os, io, re, json, time, zipfile, random, sqlite3, fnmatch, urllib.request
from dataclasses import dataclass
from typing import Callable, Optional

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

random.seed(42); np.random.seed(42)
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 100

In [ ]:
LOCAL_CSV = os.environ.get("LOCAL_CSV")   # web Sổ điểm tự đặt biến này; hoặc bạn gán đường dẫn, ví dụ "student-mat.csv"
UCI_URL = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

def _find_csv_in_zip(zbytes, target="student-mat.csv"):
    z = zipfile.ZipFile(io.BytesIO(zbytes))
    for name in z.namelist():
        if name.endswith(target):
            return z.read(name)
        if name.endswith(".zip"):
            r = _find_csv_in_zip(z.read(name), target)
            if r:
                return r
    return None

def synthetic_students(n=395, seed=42):
    """Dữ liệu tổng hợp cùng schema với UCI student-mat (chỉ dùng khi offline)."""
    rng = np.random.default_rng(seed)
    g1 = np.clip(rng.normal(10.9, 3.3, n).round(), 3, 19).astype(int)
    g2 = np.clip(g1 + rng.normal(0, 1.8, n).round(), 0, 19).astype(int)
    g3 = np.clip(g2 + rng.normal(0, 1.5, n).round(), 0, 20).astype(int)
    yn = lambda p: rng.choice(["yes", "no"], n, p=[p, 1 - p])
    return pd.DataFrame({
        "school": rng.choice(["GP", "MS"], n, p=[0.88, 0.12]),
        "sex": rng.choice(["F", "M"], n),
        "age": rng.choice([15, 16, 17, 18, 19, 20, 21, 22], n, p=[.21, .26, .25, .21, .05, .01, .005, .005]),
        "address": rng.choice(["U", "R"], n, p=[.78, .22]),
        "famsize": rng.choice(["GT3", "LE3"], n, p=[.71, .29]),
        "Pstatus": rng.choice(["T", "A"], n, p=[.9, .1]),
        "Medu": rng.integers(0, 5, n), "Fedu": rng.integers(0, 5, n),
        "Mjob": rng.choice(["teacher", "health", "services", "at_home", "other"], n),
        "Fjob": rng.choice(["teacher", "health", "services", "at_home", "other"], n),
        "reason": rng.choice(["home", "reputation", "course", "other"], n),
        "guardian": rng.choice(["mother", "father", "other"], n, p=[.7, .23, .07]),
        "traveltime": rng.integers(1, 5, n), "studytime": rng.integers(1, 5, n),
        "failures": rng.choice([0, 1, 2, 3], n, p=[.79, .12, .04, .05]),
        "schoolsup": yn(.13), "famsup": yn(.61), "paid": yn(.46), "activities": yn(.51),
        "nursery": yn(.8), "higher": yn(.95), "internet": yn(.83), "romantic": yn(.33),
        "famrel": rng.integers(1, 6, n), "freetime": rng.integers(1, 6, n), "goout": rng.integers(1, 6, n),
        "Dalc": rng.integers(1, 6, n), "Walc": rng.integers(1, 6, n), "health": rng.integers(1, 6, n),
        "absences": rng.poisson(5.7, n), "G1": g1, "G2": g2, "G3": g3,
    })

def load_students():
    if LOCAL_CSV and os.path.exists(LOCAL_CSV):
        d = pd.read_csv(LOCAL_CSV, sep=";")
        if d.shape[1] == 1:                      # không phải dấu ; thì để pandas tự dò
            d = pd.read_csv(LOCAL_CSV, sep=None, engine="python")
        return d, "local-csv"
    try:
        raw = urllib.request.urlopen(UCI_URL, timeout=20).read()
        csv_bytes = _find_csv_in_zip(raw)
        if csv_bytes:
            return pd.read_csv(io.BytesIO(csv_bytes), sep=";"), "UCI (dataset thật)"
    except Exception as e:
        print("Không tải được từ UCI:", type(e).__name__, "-", e)
    print("⚠️  Dùng dữ liệu TỔNG HỢP cùng schema (không phải dataset thật). Hãy chạy lại khi có mạng hoặc đặt LOCAL_CSV.")
    return synthetic_students(), "synthetic (cùng schema)"

df_raw, SOURCE = load_students()
print("Nguồn dữ liệu:", SOURCE, "| shape =", df_raw.shape)
df_raw.head()

In [ ]:
df = df_raw.copy().reset_index(drop=True)
df.insert(0, "sid", [f"S{i+1:04d}" for i in range(len(df))])   # khoá định danh học sinh

# Chia các trường dữ liệu thành 3 nhóm nhạy cảm khác nhau (để minh hoạ fine-grained AC)
ACADEMIC = frozenset(["G1", "G2", "G3", "failures", "absences", "studytime", "paid", "schoolsup", "higher"])
FAMILY   = frozenset(["famsize", "Pstatus", "Medu", "Fedu", "Mjob", "Fjob", "guardian", "famrel", "famsup"])
ALL_FIELDS = [c for c in df.columns if c not in ("sid", "school")]
PERSONAL = frozenset(f for f in ALL_FIELDS if f not in ACADEMIC | FAMILY)   # sex, age, health, Dalc, Walc, romantic...
GROUP_FIELDS = {"academic": ACADEMIC, "personal": PERSONAL, "family": FAMILY}
FIELD_GROUP = {f: g for g, fs in GROUP_FIELDS.items() for f in fs}

REC = {r["sid"]: r for r in df.to_dict("records")}
SCHOOLS = sorted(df.school.unique())
print("Số học sinh theo trường:", df.school.value_counts().to_dict())
print({g: len(fs) for g, fs in GROUP_FIELDS.items()}, "(số trường dữ liệu theo nhóm)")

## Phần 1 — Bốn mô hình NoSQL (§2.1)
Cùng một bộ dữ liệu học sinh được lưu theo 3 mô hình dạng **cây** và 1 mô hình **đồ thị**:

- **Key-value:** `school → key (sid) → value (blob JSON)`. Giá trị là một khối mờ.
- **Wide-column:** `column family (nhóm dữ liệu) → row (school#sid) → column → value`.
- **Document:** `collection (school) → document (sid) → field → value`.
- **Graph:** nút và cạnh (xây ở Phần 4).

In [ ]:
DB = "school_db"
KV, DOC, WC = {}, {}, {}
to_json = lambda o: json.dumps(o, default=lambda x: x.item() if hasattr(x, "item") else str(x))

for sid, r in REC.items():
    s = r["school"]
    flds = {f: r[f] for f in ALL_FIELDS}
    KV.setdefault(s, {})[sid] = to_json(flds)                       # key-value: value là blob
    DOC.setdefault(s, {})[sid] = flds                               # document
    for g, fs in GROUP_FIELDS.items():                              # wide-column: CF = nhóm dữ liệu
        WC.setdefault(g, {}).setdefault(f"{s}#{sid}", {}).update({f: flds[f] for f in fs})

sid0 = next(iter(REC)); s0 = REC[sid0]["school"]
print("KEY-VALUE  :", f"{s0}/{sid0}", "->", KV[s0][sid0][:90], "...")
print("DOCUMENT   :", f"{s0}/{sid0}", "->", {k: DOC[s0][sid0][k] for k in ["age", "health", "G1", "G3"]}, "...")
print("WIDE-COLUMN:", f"academic/{s0}#{sid0}", "->", {k: WC["academic"][f"{s0}#{sid0}"][k] for k in ["G1", "G2", "G3"]}, "...")

## Phần 2 — Cấu trúc quan hệ: cây (Fig. 5) và đồ thị có hướng (Fig. 6)
Bài báo: key-value, wide-column, document biểu diễn được bằng **cây tối đa 5 tầng** (tầng 1 = chính database). Ta dựng cây từ dữ liệu thật và đếm số nút từng tầng.

In [ ]:
def nested_to_tree(root, nested):
    """Chuyển dict lồng nhau thành cây; nút giá trị (value) nằm ở tầng ngay dưới nút chứa nó."""
    T = nx.DiGraph(); T.add_node((root,), level=1, label=root)
    def rec(path, obj, level):
        if isinstance(obj, dict):
            for k, v in obj.items():
                p = path + (str(k),)
                T.add_node(p, level=level + 1, label=str(k)); T.add_edge(path, p)
                rec(p, v, level + 1)
        else:
            p = path + (str(obj),)
            T.add_node(p, level=level + 1, label=str(obj)[:10]); T.add_edge(path, p)
    rec((root,), nested, 1)
    return T

TREES = {"Key-Value": nested_to_tree(DB, KV),
         "Wide-Column": nested_to_tree(DB, WC),
         "Document": nested_to_tree(DB, DOC)}

rows = []
for name, T in TREES.items():
    lv = pd.Series([d["level"] for _, d in T.nodes(data=True)]).value_counts().sort_index()
    rows.append({"model": name, **{f"Tầng {l}": int(c) for l, c in lv.items()}})
levels_df = pd.DataFrame(rows).set_index("model").fillna(0).astype(int)
print("Số nút theo tầng (tầng 1 = database). Key-value dừng ở tầng 4 (không có tầng 5), đúng như bài báo.")
levels_df

In [ ]:
# Vẽ minh hoạ cây (giống Fig. 5) trên một mẫu nhỏ
gp_ids = [sid for sid, r in REC.items() if r["school"] == SCHOOLS[0]][:2]
ot_ids = [sid for sid, r in REC.items() if r["school"] == SCHOOLS[-1]][:1]
sample = {}
for sid in gp_ids + ot_ids:
    s = REC[sid]["school"]
    sample.setdefault(s, {})[sid] = {k: REC[sid][k] for k in ["G1", "G3", "health"]}
T = nested_to_tree(DB, sample)

pos = nx.multipartite_layout(T, subset_key="level", align="horizontal")
pos = {n: (x, -y) for n, (x, y) in pos.items()}
fig, ax = plt.subplots(figsize=(11, 5))
nx.draw(T, pos, ax=ax, labels={n: d["label"] for n, d in T.nodes(data=True)},
        node_color=[d["level"] for _, d in T.nodes(data=True)], cmap=plt.cm.Pastel1,
        node_size=1100, font_size=8, arrows=True)
ax.set_title("Document model dạng cây: DB → collection(school) → document(sid) → field → value")
plt.show()

## Phần 3 — Luật ABAC từ cấu trúc cây (§3.2.1–3.2.2)
**Ý tưởng của bài báo:** thuộc tính của subject và object chính là **các nút trên đường đi trong cây**. Một luật có thể chỉ dùng **một nút** (không cần đủ các tầng) và khi đó **các nút cấp thấp hơn thừa hưởng quyền** của nút bên trên.

Policy là một tuple: `{database, thuộc tính subject, nút lá subject (tuỳ chọn), thuộc tính object, nút lá object (tuỳ chọn), actions, permission}`.

Áp dụng vào trường học:

- **Cây subject:** `DB → school (tổ chức) → department → group → user`
- **Cây object (document model):** `DB → school (collection) → sid (document) → field`

Các luật mẫu (có thêm *điều kiện* như §3.3 và §4.7: phụ thuộc giá trị dữ liệu và môi trường):

| Luật | Nội dung |
|---|---|
| R1 | Giáo viên Math/Portuguese của trường *s* đọc nhóm *academic* của học sinh trường *s*, **nếu đã tập huấn bảo mật** |
| R2 | Tư vấn viên đọc nhóm *personal* |
| R3 | Tư vấn viên đọc *academic* **chỉ của học sinh có nguy cơ** (G2 < 10) |
| R4 | Quản trị đọc mọi trường dữ liệu |
| R5 | Quản trị cập nhật nhóm *family* |
| R6 | Học sinh đọc *academic* **của chính mình** |
| R7 | **DENY**: học sinh không đọc G3 khi kỳ chấm điểm chưa chốt (điều kiện môi trường) |

In [ ]:
# ---- Người dùng hệ thống (MÔ PHỎNG; dataset không có) ----
DEPTS = {"Math": ["Math-A", "Math-B"], "Portuguese": ["Port-A"], "Counseling": ["Tutors"], "Admin": ["Registrar"]}
ROLE_OF = {"Math": "teacher", "Portuguese": "teacher", "Counseling": "counselor", "Admin": "admin"}
STAFF = []
for s in SCHOOLS:
    for dept, groups in DEPTS.items():
        for g in groups:
            for i in range(2):
                STAFF.append({"user": f"{s.lower()}_{g.lower().replace('-', '')}_{i+1}", "school": s, "dept": dept,
                              "group": g, "role": ROLE_OF[dept], "trained": (i == 0)})  # người thứ 2 chưa tập huấn
STUDENTS = [{"user": sid, "school": r["school"], "dept": "Students", "group": f"age{int(r['age'])}",
             "role": "student", "trained": True} for sid, r in REC.items()]
SUBJECTS = {u["user"]: u for u in STAFF + STUDENTS}
subject_path = lambda s: (DB, s["school"], s["dept"], s["group"], s["user"])   # tầng 1..5
print(len(STAFF), "nhân sự mô phỏng +", len(STUDENTS), "học sinh làm subject")
print("Ví dụ path subject:", subject_path(SUBJECTS[f"{SCHOOLS[0].lower()}_matha_1"]))
pd.DataFrame(STAFF).head(6)

In [ ]:
@dataclass
class Rule:
    rid: str
    subj: dict            # {tầng: giá trị | tập giá trị | pattern | "*"}  (chỉ cần nêu các tầng liên quan)
    obj: dict
    actions: tuple
    permit: bool = True
    cond: Optional[Callable] = None
    note: str = ""

def spec_match(spec, path):
    """Một luật chỉ cần 1 nút hoặc vài nút bất kỳ trên đường đi (không cần đủ các tầng)."""
    for lvl, want in spec.items():
        if lvl - 1 >= len(path):
            return False
        val = path[lvl - 1]
        if want == "*":
            continue
        if isinstance(want, (set, frozenset, list, tuple)):
            if val not in want: return False
        elif isinstance(want, str) and any(c in want for c in "*?"):
            if not fnmatch.fnmatch(str(val), want): return False
        elif val != want:
            return False
    return True

# --- điều kiện (dữ liệu / môi trường) ---
at_risk = lambda ctx: ctx["record"]["G2"] < 10
is_trained = lambda ctx: bool(ctx["subject"]["trained"])
is_self = lambda ctx: ctx["subject"]["user"] == ctx["record"]["sid"]
grading_not_closed = lambda ctx: not ctx["env"].get("grading_period_closed", False)
for _f, _n in [(at_risk, "at_risk"), (is_trained, "is_trained"), (is_self, "is_self"), (grading_not_closed, "grading_not_closed")]:
    _f.__name__ = _n   # đặt tên để hiển thị trong bảng luật

def make_rules(schools):
    R = []
    for s in schools:
        R += [
            Rule(f"R1-{s}", {2: s, 3: {"Math", "Portuguese"}}, {2: s, 4: ACADEMIC}, ("read",), True, is_trained, "GV đã tập huấn đọc academic"),
            Rule(f"R2-{s}", {2: s, 3: "Counseling"}, {2: s, 4: PERSONAL}, ("read",), True, None, "Tư vấn đọc personal"),
            Rule(f"R3-{s}", {2: s, 3: "Counseling"}, {2: s, 4: ACADEMIC}, ("read",), True, at_risk, "Tư vấn đọc academic của HS nguy cơ"),
            Rule(f"R4-{s}", {2: s, 3: "Admin"}, {2: s, 4: "*"}, ("read",), True, None, "Quản trị đọc tất cả"),
            Rule(f"R5-{s}", {2: s, 3: "Admin"}, {2: s, 4: FAMILY}, ("update",), True, None, "Quản trị cập nhật family"),
            Rule(f"R6-{s}", {2: s, 3: "Students"}, {2: s, 4: ACADEMIC}, ("read",), True, is_self, "HS đọc academic của mình"),
            Rule(f"R7-{s}", {2: s, 3: "Students"}, {2: s, 4: {"G3"}}, ("read",), False, grading_not_closed, "DENY G3 khi chưa chốt điểm"),
        ]
    return R
RULES = make_rules(SCHOOLS)

AUDIT = []
def decide(subject, action, sid, field, env=None, rules=None, log=False):
    """PDP: default-deny, deny-overrides."""
    env = env or {}; rules = RULES if rules is None else rules
    rec = REC[sid]; path = (DB, rec["school"], sid, field); spath = subject_path(subject)
    ctx = {"subject": subject, "record": rec, "env": env}
    allowed, by = False, "default-deny"
    for r in rules:
        if action in r.actions and spec_match(r.subj, spath) and spec_match(r.obj, path) and (r.cond is None or r.cond(ctx)):
            if not r.permit:
                allowed, by = False, r.rid; break
            if not allowed:
                allowed, by = True, r.rid
    if log:
        AUDIT.append({"user": subject["user"], "role": subject["role"], "school": subject["school"], "action": action,
                      "sid": sid, "field": field, "group": FIELD_GROUP[field],
                      "decision": "permit" if allowed else "deny", "rule": by})
    return allowed, by

def read_record(subject, sid, env=None, rules=None, log=False):
    """Fine-grained: lọc từng trường theo luật."""
    return {f: REC[sid][f] for f in ALL_FIELDS if decide(subject, "read", sid, f, env, rules, log)[0]}

def fmt(v):
    for g, fs in GROUP_FIELDS.items():
        if v == fs: return f"<{g} fields>"
    return sorted(v) if isinstance(v, (set, frozenset)) else v
pd.DataFrame([{"rule": r.rid, "subject attrs": {f"L{k}": fmt(v) for k, v in r.subj.items()},
               "object attrs": {f"L{k}": fmt(v) for k, v in r.obj.items()}, "actions": r.actions,
               "permit": r.permit, "condition": getattr(r.cond, "__name__", None) if r.cond else None}
              for r in RULES if r.rid.endswith(SCHOOLS[0])])

In [ ]:
S = SCHOOLS[0]
stu = next(sid for sid, r in REC.items() if r["school"] == S)
other_school_stu = next((sid for sid, r in REC.items() if r["school"] != S), stu)
def show(user, action, sid, field, env=None):
    ok, by = decide(SUBJECTS[user], action, sid, field, env)
    print(f"{'PERMIT' if ok else 'DENY  '} | {user:14s} {action:6s} {sid}.{field:8s} env={env or {}} -> {by}")

t_ok, t_no, cns, adm = f"{S.lower()}_matha_1", f"{S.lower()}_matha_2", f"{S.lower()}_tutors_1", f"{S.lower()}_registrar_1"
show(t_ok, "read", stu, "G3")                  # GV đã tập huấn đọc điểm
show(t_no, "read", stu, "G3")                  # GV chưa tập huấn -> deny
show(t_ok, "read", stu, "health")              # GV không có quyền nhóm personal
show(t_ok, "read", other_school_stu, "G3")     # GV khác trường -> deny (nếu có học sinh trường khác)
show(cns, "read", stu, "health")               # tư vấn đọc personal
show(cns, "read", stu, "G3")                   # tư vấn đọc G3 chỉ khi HS nguy cơ
print("   học sinh", stu, "G2 =", REC[stu]["G2"], "-> at_risk =", REC[stu]["G2"] < 10)
show(adm, "update", stu, "Mjob")               # quản trị cập nhật family
show(adm, "update", stu, "G3")                 # quản trị không được sửa điểm
show(stu, "read", stu, "G1")                   # HS đọc điểm của mình
show(stu, "read", stu, "G3", {"grading_period_closed": False})   # R7 deny
show(stu, "read", stu, "G3", {"grading_period_closed": True})    # đã chốt -> permit

### Cùng ý tưởng trên Wide-column: nhóm dữ liệu là tầng 2 (column family), trường học nằm trong khoá hàng
Ở wide-column, cùng một thuộc tính nằm ở tầng khác. Luật chỉ cần đổi tầng và dùng pattern trên row key (`GP#*`).
#### Giới hạn của key-value (§3.2.2)
Giá trị là blob mờ, nên luật chỉ chạm được tới tầng khoá (cả học sinh). Muốn kiểm soát theo nhóm trường phải **dựng thêm nhánh ngang** (tách khoá `sid#group`), làm số khoá và độ phức tạp tăng lên.

In [ ]:
rule_wc = Rule("WC-R1", {2: S, 3: {"Math"}}, {2: "academic", 3: f"{S}#*"}, ("read",))
other = other_school_stu
print("Wide-column: path (academic, GP#S…, G3)     ->", spec_match(rule_wc.obj, (DB, "academic", f"{S}#{stu}", "G3")))
print("Wide-column: path (personal, GP#S…, health) ->", spec_match(rule_wc.obj, (DB, "personal", f"{S}#{stu}", "health")))
print("Wide-column: trường khác                    ->", spec_match(rule_wc.obj, (DB, "academic", f"{REC[other]['school']}#{other}", "G3")) if REC[other]["school"] != S else "(không có trường khác trong mẫu)")

# Key-value: blob mờ -> rule chỉ ở mức khoá; phải tách khoá để có granularity nhóm
KV_SPLIT = {}
for sid, r in REC.items():
    for g, fs in GROUP_FIELDS.items():
        KV_SPLIT.setdefault(r["school"], {})[f"{sid}#{g}"] = to_json({f: r[f] for f in fs})
n_kv = sum(len(v) for v in KV.values()); n_split = sum(len(v) for v in KV_SPLIT.values())
print(f"\nKey-value gốc: {n_kv} khoá | sau khi tách theo nhóm: {n_split} khoá (x{n_split/n_kv:.0f}) -> thêm 1 tầng ngang, quản trị phức tạp hơn")
print("Số trường lộ ra khi cấp quyền ở mức khoá gốc:", len(json.loads(KV[S][stu])), "/", len(ALL_FIELDS))

## Phần 4 — Graph model (§3.2.3, Fig. 9–10)
Trong graph model:

- Thuộc tính là **chuỗi cạnh/nút**, không giới hạn số tầng và có thể là DAG hay có chu trình.
- **Cạnh có thể là hành động được phép**, nên luật nằm ngay trong database dưới dạng chuỗi liên kết (Fig. 10).
- **RBAC** dùng bằng cách coi một số nút là *role*.
- Điều kiện phụ thuộc dữ liệu có thể **vật chất hoá thành cạnh** (ví dụ cạnh tới nút `academic-atrisk`).

In [ ]:
def build_policy_graph():
    G = nx.DiGraph()
    add = lambda u, v, lab: G.add_edge(u, v, label=lab)
    # --- phía subject: user -member_of-> group -member_of-> dept -works_for-> school ; user -has_role-> role
    for u in STAFF:
        s = u["school"]
        add(u["user"], f"group:{s}:{u['group']}", "member_of")
        add(f"group:{s}:{u['group']}", f"dept:{s}:{u['dept']}", "member_of")
        add(f"dept:{s}:{u['dept']}", f"school:{s}", "works_for")
        if u["role"] != "teacher" or u["trained"]:        # GV chưa tập huấn không được gán role
            add(u["user"], f"role:{u['role']}@{s}", "has_role")
    # --- cạnh hành động = luật nhúng trong DB (RBAC qua nút role)
    for s in SCHOOLS:
        add(f"role:teacher@{s}", f"scope:{s}:academic", "can_read")
        add(f"role:counselor@{s}", f"scope:{s}:personal", "can_read")
        add(f"role:counselor@{s}", f"scope:{s}:academic-atrisk", "can_read")
        add(f"role:admin@{s}", f"scope:{s}:all", "can_read")
    # --- phía object: field -belongs_to-> scope (DAG: một field thuộc nhiều scope)
    for sid, r in REC.items():
        s = r["school"]
        for f in ALL_FIELDS:
            fo, g = f"{sid}/{f}", FIELD_GROUP[f]
            add(fo, f"scope:{s}:{g}", "belongs_to"); add(fo, f"scope:{s}:all", "belongs_to")
            add(fo, f"{sid}:{g}", "belongs_to")
            if g == "academic" and r["G2"] < 10:          # điều kiện dữ liệu -> vật chất hoá thành cạnh
                add(fo, f"scope:{s}:academic-atrisk", "belongs_to")
        add(sid, f"{sid}:academic", "can_read")           # học sinh đọc academic của chính mình
    return G

PG = build_policy_graph()
print("Policy graph:", PG.number_of_nodes(), "nút,", PG.number_of_edges(), "cạnh")

SUBJ_LABELS, OBJ_LABELS = {"member_of", "works_for", "has_role"}, {"belongs_to"}
def climb(G, start, labels):
    seen, stack = {start}, [start]
    while stack:
        n = stack.pop()
        for _, m, d in G.out_edges(n, data=True):
            if d["label"] in labels and m not in seen:
                seen.add(m); stack.append(m)
    return seen

def graph_decide(G, user, action, obj_node):
    subj_attrs, obj_attrs = climb(G, user, SUBJ_LABELS), climb(G, obj_node, OBJ_LABELS)
    for s in subj_attrs:
        for _, o, d in G.out_edges(s, data=True):
            if d["label"] == f"can_{action}" and o in obj_attrs:
                return True, (s, o)
    return False, None

print(graph_decide(PG, t_ok, "read", f"{stu}/G3"))
print(graph_decide(PG, t_no, "read", f"{stu}/G3"))
print(graph_decide(PG, stu, "read", f"{stu}/G1"))
print(graph_decide(PG, stu, "read", f"{stu}/health"))

In [ ]:
# Vẽ đồ thị con (giống Fig. 9/10): chuỗi subject -> cạnh can_read -> chuỗi object
nodes = climb(PG, t_ok, SUBJ_LABELS)
nodes |= {o for s in list(nodes) for _, o, d in PG.out_edges(s, data=True) if d["label"] == "can_read"}
for f in ["G1", "G3"]:
    nodes |= climb(PG, f"{stu}/{f}", OBJ_LABELS)
H = PG.subgraph(nodes)
# chỉ giữ scope academic cho gọn
H = H.subgraph([n for n in H.nodes if not (n.startswith("scope:") and not n.endswith("academic"))])
pos = nx.spring_layout(H, seed=3, k=1.2)
fig, ax = plt.subplots(figsize=(12, 6))
nx.draw(H, pos, ax=ax, with_labels=True, node_size=1800, font_size=7, node_color="#cfe3f5", arrows=True)
nx.draw_networkx_edge_labels(H, pos, ax=ax, edge_labels={(u, v): d["label"] for u, v, d in H.edges(data=True)}, font_size=7)
ax.set_title("Graph model: user → group → dept → school | role —can_read→ scope ← belongs_to — field")
plt.show()

In [ ]:
# Kiểm chứng: ABAC trên cây (Phần 3) và luật nhúng trong đồ thị (Phần 4) cho cùng quyết định
ENV_CLOSED = {"grading_period_closed": True}
rr = random.Random(3); sids = list(REC); cases = []
for u in STAFF:
    for _ in range(40): cases.append((u["user"], rr.choice(sids), rr.choice(ALL_FIELDS)))
for sid in rr.sample(sids, min(100, len(sids))):
    cases.append((sid, sid, rr.choice(ALL_FIELDS))); cases.append((sid, rr.choice(sids), rr.choice(ALL_FIELDS)))
agree = sum(decide(SUBJECTS[u], "read", sid, f, env=ENV_CLOSED)[0] == graph_decide(PG, u, "read", f"{sid}/{f}")[0]
            for u, sid, f in cases)
n_permit = sum(decide(SUBJECTS[u], "read", sid, f, env=ENV_CLOSED)[0] for u, sid, f in cases)
print(f"Đồng nhất {agree}/{len(cases)} quyết định ({100*agree/len(cases):.1f}%), trong đó {n_permit} permit")

## Phần 5 — Các lưu ý khi triển khai (§4)
### 4.1 Fine-grained access control
Nhiều NoSQL chỉ cấp quyền ở mức database/document (**coarse-grained**), tức là có quyền hoặc không. Ta so sánh với FGAC theo từng trường.

In [ ]:
def coarse_read(subject, sid):
    """Mô phỏng ủy quyền mức database/document: có quyền 'read' là thấy cả document."""
    if subject["role"] in ("teacher", "counselor", "admin") and subject["school"] == REC[sid]["school"]:
        return {f: REC[sid][f] for f in ALL_FIELDS}
    return {}

sample_ids = [sid for sid, r in REC.items() if r["school"] == S][:100]
rows = []
for label, u in [("teacher (đã tập huấn)", t_ok), ("counselor", cns), ("admin", adm)]:
    sj = SUBJECTS[u]
    coarse = sum(len(coarse_read(sj, sid)) for sid in sample_ids) / len(sample_ids)
    fine = sum(len(read_record(sj, sid, ENV_CLOSED)) for sid in sample_ids) / len(sample_ids)
    rows.append({"vai trò": label, "trường/HS (coarse)": coarse, "trường/HS (FGAC)": fine, "lộ thừa": coarse - fine})
fg = pd.DataFrame(rows).set_index("vai trò")
print(fg.round(1))
fg[["trường/HS (coarse)", "trường/HS (FGAC)"]].plot.bar(figsize=(7, 3.5), rot=0, title="Số trường dữ liệu mỗi học sinh mà vai trò nhìn thấy")
plt.tight_layout(); plt.show()

### 4.2 Security: NoSQL injection
Bài báo nêu NoSQL cũng bị injection, ví dụ toán tử `$where`. Dưới đây là mô phỏng an toàn trong bộ nhớ (không có DB thật) về **operator injection**: ứng dụng nhận JSON từ client và đưa thẳng vào query.

In [ ]:
LOGINS = [{"user": "gv_an", "pin": "4821", "role": "teacher"}, {"user": "admin_ha", "pin": "9937", "role": "admin"}]

def mini_match(doc, query):
    """Bộ so khớp truy vấn kiểu MongoDB rút gọn: hỗ trợ $ne, $gt, $regex."""
    for k, cond in query.items():
        v = doc.get(k)
        if isinstance(cond, dict):
            for op, arg in cond.items():
                if op == "$ne" and not (v != arg): return False
                elif op == "$gt" and not (v is not None and v > arg): return False
                elif op == "$regex" and not re.search(arg, str(v)): return False
                elif op not in ("$ne", "$gt", "$regex"): raise ValueError(op)
        elif v != cond:
            return False
    return True
find_one = lambda coll, q: next((d for d in coll if mini_match(d, q)), None)

def login_vulnerable(body_json):
    b = json.loads(body_json)
    return find_one(LOGINS, {"user": b["user"], "pin": b["pin"]})        # tin tưởng kiểu dữ liệu client gửi

def login_safe(body_json):
    b = json.loads(body_json)
    if not (isinstance(b.get("user"), str) and isinstance(b.get("pin"), str)):   # ép kiểu: chặn object/toán tử
        return None
    return find_one(LOGINS, {"user": b["user"], "pin": b["pin"]})

normal = '{"user": "admin_ha", "pin": "0000"}'
attack = '{"user": "admin_ha", "pin": {"$ne": null}}'
print("Đăng nhập sai PIN bình thường :", login_vulnerable(normal))
print("Tấn công {'$ne': null} (vuln) :", login_vulnerable(attack))
print("Tấn công {'$ne': null} (safe) :", login_safe(attack))

### 4.3 Query language
Không có ngôn ngữ truy vấn chuẩn, nên khó viết một bộ thực thi FGAC chung. Cùng một yêu cầu *"điểm của học sinh có G3 < 10"* ở các hệ khác nhau:

| Mô hình | Ví dụ truy vấn | Ghi chú cho access control |
|---|---|---|
| Document (kiểu MongoDB) | `db.GP.find({"G3": {"$lt": 10}}, {"G1":1,"G3":1})` | Có projection, lọc được theo trường |
| Wide-column (kiểu CQL) | `SELECT G1, G3 FROM academic WHERE school='GP' AND G3 < 10 ALLOW FILTERING;` | Quyền thường gắn với bảng/column family |
| Key-value (kiểu Redis) | `GET GP:S0001` | Không lọc được theo giá trị, phải đọc cả blob |
| Graph (kiểu Cypher) | `MATCH (s:Student)-[:HAS]->(a:Academic) WHERE a.G3 < 10 RETURN s, a` | Quyền gắn với nút/cạnh |

(Các câu trên chỉ minh hoạ khái niệm, không chạy trong notebook.)

### 4.4 Data consistency
ABAC/RBAC động cần trạng thái truy cập **hiện tại và nhất quán**. NoSQL phân tán thường chỉ **eventually consistent**. Mô phỏng: thu hồi quyền của giáo viên ở primary nhưng replica chưa đồng bộ.

In [ ]:
class ReplicatedStore:
    """Primary + replica đồng bộ trễ (eventual consistency)."""
    def __init__(self): self.primary, self.replica, self.pending = {}, {}, []
    def write(self, k, v): self.primary[k] = v; self.pending.append((k, v))
    def sync(self):
        for k, v in self.pending: self.replica[k] = v
        self.pending.clear()
    def read(self, k, strong=False): return (self.primary if strong else self.replica).get(k)

ROLES = ReplicatedStore()
def pdp_read_grades(user, strong=False):
    return "PERMIT" if ROLES.read(user, strong) == "teacher" else "DENY"

timeline = []
ROLES.write("gv_an", "teacher"); ROLES.sync()
timeline.append(("t0: cấp quyền teacher + đồng bộ", pdp_read_grades("gv_an"), pdp_read_grades("gv_an", True)))
ROLES.write("gv_an", "revoked")                                    # thu hồi quyền ở primary
timeline.append(("t1: thu hồi ở primary (replica chưa sync)", pdp_read_grades("gv_an"), pdp_read_grades("gv_an", True)))
ROLES.sync()
timeline.append(("t2: replica đã đồng bộ", pdp_read_grades("gv_an"), pdp_read_grades("gv_an", True)))
pd.DataFrame(timeline, columns=["thời điểm", "đọc eventual (replica)", "đọc strong (primary)"])

Ở thời điểm **t1**, đọc từ replica vẫn **PERMIT** dù quyền đã bị thu hồi. Đây là khoảng hở mà bài báo cảnh báo. Cách giảm thiểu thường dùng là đọc *strong/quorum* riêng cho dữ liệu chính sách và thu hồi quyền, đánh đổi bằng hiệu năng (4.5).
### 4.5 Performance
Access control luôn có chi phí. Ta đo thời gian đọc 100 hồ sơ: không AC, FGAC "ngây thơ", và FGAC có **lọc trước luật theo subject**.

In [ ]:
def bench(fn, n=5):
    fn(); t = time.perf_counter()
    for _ in range(n): fn()
    return (time.perf_counter() - t) / n * 1000

sj = SUBJECTS[t_ok]
applicable = [r for r in RULES if spec_match(r.subj, subject_path(sj))]      # tối ưu: lọc luật theo subject một lần
res = {
    "Không AC": bench(lambda: [DOC[S][sid] for sid in sample_ids]),
    "FGAC (duyệt mọi luật)": bench(lambda: [read_record(sj, sid, ENV_CLOSED) for sid in sample_ids]),
    "FGAC (lọc luật trước)": bench(lambda: [read_record(sj, sid, ENV_CLOSED, rules=applicable) for sid in sample_ids]),
}
perf = pd.Series(res, name="ms / 100 hồ sơ").to_frame()
perf["ms / hồ sơ"] = perf.iloc[:, 0] / len(sample_ids)
print(perf.round(2))
perf.iloc[:, 0].plot.bar(figsize=(6, 3.2), rot=15, title="Chi phí access control (ms / 100 hồ sơ)")
plt.tight_layout(); plt.show()

### 4.6 Audit
Các công cụ giám sát NoSQL thường tập trung vào hiệu năng và ít hỗ trợ audit truy cập. Ở đây PDP ghi log mọi quyết định, rồi ta phân tích bằng pandas.

In [ ]:
AUDIT.clear()
rr = random.Random(5); fields_rw = ALL_FIELDS
for _ in range(3000):
    a = rr.choice(STAFF) if rr.random() < 0.6 else rr.choice(STUDENTS)
    sid = a["user"] if (a["role"] == "student" and rr.random() < 0.5) else rr.choice(list(REC))
    decide(a, "read" if rr.random() < 0.85 else "update", sid, rr.choice(fields_rw),
           env={"grading_period_closed": rr.random() < 0.5}, log=True)
audit = pd.DataFrame(AUDIT)
print(audit.groupby(["role", "decision"]).size().unstack(fill_value=0))
print("\nQuy tắc quyết định nhiều nhất:\n", audit.rule.str.replace(r"-.*", "", regex=True).value_counts().head(8))
audit.groupby(["role", "decision"]).size().unstack(fill_value=0).plot.bar(stacked=True, figsize=(7, 3.5), rot=0, title="Audit: quyết định theo vai trò")
plt.tight_layout(); plt.show()
audit[(audit.decision == "deny") & (audit.role.isin(["teacher", "counselor"]))].head()

### 4.7 Environment conditions control
Điều kiện môi trường có thể lấy từ chính NoSQL (phải xét consistency, 4.4 và performance, 4.5) hoặc từ nguồn ngoài (phải bảo mật nguồn đó, 4.2). Ở luật **R7**, điều kiện *"kỳ chấm điểm đã chốt"* là điều kiện môi trường:

In [ ]:
rows = []
for closed in (False, True):
    env = {"grading_period_closed": closed}
    rows.append({"grading_period_closed": closed,
                 "HS đọc G1 (điểm giữa kỳ)": decide(SUBJECTS[stu], "read", stu, "G1", env)[0],
                 "HS đọc G3 (điểm cuối kỳ)": decide(SUBJECTS[stu], "read", stu, "G3", env)[0],
                 "GV đã tập huấn đọc G3": decide(SUBJECTS[t_ok], "read", stu, "G3", env)[0]})
pd.DataFrame(rows).set_index("grading_period_closed")

### 4.8 Support for AI: từ văn bản tự nhiên sang luật truy cập
Bài báo mô tả quy trình NLP: nhận diện câu có chứa luật → từ điển → ngữ pháp luật hình thức (các bước ở chú thích 1: tách câu, tách từ, nhận dạng thực thể, nhận dạng quan hệ). Dưới đây là phiên bản **rule-based tối giản** để minh hoạ quy trình, **không phải mô hình AI thật**. NoSQL (đặc biệt graph) là nơi lý tưởng để lưu quan hệ phân cấp mà bước này cần.

In [ ]:
POLICY_TEXT = ("Teachers from the Math department at school GP can read academic data if they have attended privacy training. "
               "Counselors from the Counseling department at school GP can read personal data. "
               "Teachers from the Math department at school GP cannot read family data. "
               "The school canteen opens at noon.")

COND_MAP = {"they have attended privacy training": ("attended privacy training", is_trained),
            "the student is at risk": ("student at risk", at_risk)}
PAT = re.compile(r"(?P<role>teachers|counselors|admins)\s+(?:from|in)\s+the\s+(?P<dept>\w+)\s+department\s+(?:at|of)\s+school\s+(?P<school>\w+)\s+"
                 r"(?P<modal>can|cannot)\s+(?P<action>read|update)\s+(?P<group>academic|personal|family)\s+data(?:\s+if\s+(?P<cond>.+?))?\.?$", re.I)

segment = lambda t: [s.strip() for s in re.split(r"(?<=[.!?])\s+", t.strip()) if s.strip()]      # bước 1: tách câu
tokenize = lambda s: re.findall(r"[A-Za-z0-9\-]+|[^\sA-Za-z0-9]", s)                             # bước 2: tách từ

def parse_sentence(sent, i):
    m = PAT.match(sent)                                                                          # bước 3-4: nhận diện thực thể + quan hệ
    if not m: return None
    g = {k: (v.lower() if k in ("role", "modal", "action", "group", "cond") and v else v) for k, v in m.groupdict().items()}
    cname, cfn = COND_MAP.get(g["cond"], ("none", None)) if g["cond"] else ("none", None)
    rule = Rule(f"NLP-{i}", {2: g["school"], 3: g["dept"]}, {2: g["school"], 4: GROUP_FIELDS[g["group"]]},
                (g["action"],), g["modal"] == "can", cfn, sent)
    formal = (f"(user attribute = {g['role']} from {g['dept']} @ {g['school']}) ∧ (operation = {g['action']}) ∧ "
              f"(resource attribute = {g['group']} data of {g['school']}) ∧ (condition = {cname}) → "
              f"(permission = {'grant' if g['modal'] == 'can' else 'deny'})")
    return rule, formal

nlp_rules = []
for i, sent in enumerate(segment(POLICY_TEXT), 1):
    out = parse_sentence(sent, i)
    print(f"[{i}] {sent}\n    tokens: {tokenize(sent)[:8]}...")
    if out: nlp_rules.append(out[0]); print("    ->", out[1])
    else: print("    -> (không phải câu chứa luật truy cập, bỏ qua)")

print("\nKiểm thử các luật vừa sinh ra:")
for u, f in [(t_ok, "G3"), (t_no, "G3"), (t_ok, "Mjob"), (cns, "health")]:
    ok, by = decide(SUBJECTS[u], "read", stu, f, rules=nlp_rules)
    print(f"  {u:14s} read {f:7s} -> {'PERMIT' if ok else 'DENY'} ({by})")

### 4.9 Kết hợp RDBMS
Một kiến trúc lai: RDBMS giữ **người dùng, vai trò** (truy vấn "ai có quyền gì" dễ, §3.2 nói khó làm trên NoSQL), còn NoSQL giữ **dữ liệu học sinh** quy mô lớn.

In [ ]:
con = sqlite3.connect(":memory:")
pd.DataFrame(STAFF).rename(columns={"group": "grp"}).assign(trained=lambda d: d.trained.astype(int)).to_sql("staff", con, index=False)

def subject_from_sql(user):
    row = con.execute("SELECT user, school, dept, grp, role, trained FROM staff WHERE user=?", (user,)).fetchone()
    return {"user": row[0], "school": row[1], "dept": row[2], "group": row[3], "role": row[4], "trained": bool(row[5])}

print("SQL — thống kê nhân sự theo vai trò và tập huấn (truy vấn quản trị rất tự nhiên trên RDBMS):")
print(pd.read_sql("SELECT school, role, COUNT(*) AS n, SUM(trained) AS trained FROM staff GROUP BY school, role", con))

sj = subject_from_sql(t_ok)                           # subject lấy từ RDBMS
rec_view = read_record(sj, stu, ENV_CLOSED)           # dữ liệu lấy từ NoSQL (document store), lọc bằng FGAC
print("\nGV từ RDBMS đọc hồ sơ trong NoSQL:", rec_view)

## Phần 6 — Phụ lục A: Federation of Access Control
Mô hình ba dịch vụ trong liên minh:

- **RP** (Resource Provider) giữ dữ liệu.
- **RM** (Resource Manager) định vị tài nguyên.
- **RC** (Resource Consumer) là ứng dụng khách.

Áp vào giáo dục: mỗi trường là một RP, Sở giáo dục khu vực là RM, cổng giáo viên và app phụ huynh là RC. Mỗi luật là một **quan hệ nhị phân** `Rule x = {(SX, SY)}`, lưu trong graph NoSQL dưới dạng triple `SX —rule x→ SY`. Có thể **gộp luật bằng phép toán tập hợp và bao đóng**.

In [ ]:
PRIMARY = {"RP_GP": "GP", "RP_MS": "MS"}                      # RP_Mirror: bản sao của cả hai
RP_POLICY = {rp: {"share": {"academic-aggregate"}, "nonshare": {"personal", "family"}} for rp in ["RP_GP", "RP_MS", "RP_Mirror"]}  # p1, p2
RM_LINKS = {"RM_1": ["RP_GP", "RP_Mirror"], "RM_2": ["RP_MS"]}   # m1: danh sách RP tin cậy
RM_PEERS = {"RM_1": ["RM_2"], "RM_2": ["RM_1"]}                 # m2: danh sách RM tin cậy (n:n)
Credential = {("RP_GP", "RP_Mirror"), ("RP_MS", "RP_Mirror")}   # m3: S1 đáng tin hơn S2
Replace    = {("RP_Mirror", "RP_MS"), ("RP_Mirror", "RP_GP")}   # m4: yêu cầu S1 nếu S2 không sẵn sàng
Reference  = {("RP_Mirror", "RP_GP"), ("RP_Mirror", "RP_MS")}   # m5: A được tạo thành từ B, C
MAX_AGE_DAYS = 10                                               # c3: không tin thông tin cũ hơn 10 ngày

# Gộp luật bằng phép toán tập hợp + bao đóng bắc cầu
TRUSTED = set(PRIMARY) | {"RP_Mirror"}
REPLACE_EFF = {(a, b) for (a, b) in Replace if a in TRUSTED and b in TRUSTED}
closure = set(nx.transitive_closure(nx.DiGraph(list(REPLACE_EFF))).edges())
print("Replace hiệu lực (giao với tập RP tin cậy):", sorted(REPLACE_EFF))
print("Bao đóng bắc cầu của Replace:", sorted(closure))

# Lưu dạng triple trong graph NoSQL:  SX --rule--> SY
FG = nx.MultiDiGraph()
for name, rel in [("Credential", Credential), ("Replace", Replace), ("Reference", Reference)]:
    for a, b in rel: FG.add_edge(a, b, label=name)
for rm, rps in RM_LINKS.items():
    for rp in rps: FG.add_edge(rm, rp, label="trusts")
fig, ax = plt.subplots(figsize=(8, 4.5))
pos = nx.spring_layout(FG, seed=2, k=1.5)
nx.draw(FG, pos, ax=ax, with_labels=True, node_size=2200, font_size=8, node_color="#e8dff5", connectionstyle="arc3,rad=0.15")
nx.draw_networkx_edge_labels(nx.DiGraph(FG), pos, ax=ax, edge_labels={(u, v): d["label"] for u, v, d in nx.DiGraph(FG).edges(data=True)}, font_size=7)
ax.set_title("Triple luật liên minh trong graph NoSQL"); plt.show()

In [ ]:
def rp_answer(rp, topic, school, mirror_age=3):
    """RP trả lời theo luật share (p1) / non-share (p2). Chỉ chia sẻ số liệu tổng hợp."""
    pol = RP_POLICY[rp]
    if topic in pol["nonshare"]:
        raise PermissionError(f"{rp} từ chối chia sẻ '{topic}' (luật p2 non-share)")
    sub = df[df.school == school]
    return {"rp": rp, "school": school, "n": int(len(sub)), "mean_G3": round(float(sub.G3.mean()), 2),
            "age_days": mirror_age if rp == "RP_Mirror" else 0}

def find_rp(rm, school, visited=None):
    visited = (visited or []) + [rm]
    for rp in RM_LINKS[rm]:
        if PRIMARY.get(rp) == school: return rp, visited
    for peer in RM_PEERS[rm]:
        if peer not in visited:
            r = find_rp(peer, school, visited)
            if r[0]: return r
    return None, visited

def federated_query(rc, school, topic="academic-aggregate", down=(), mirror_age=3, home_rm="RM_1"):
    log = [f"{rc} → {home_rm}: yêu cầu '{topic}' của trường {school}"]
    rp, route = find_rp(home_rm, school)
    if rp is None: return {"ok": False, "log": log + ["không tìm thấy RP"]}
    log.append("Scenario 1: RM chuyển thẳng tới RP" if len(route) == 1 else f"Scenario 3: khám phá tài nguyên qua {' → '.join(route)}")
    use = rp
    if rp in down:
        subs = sorted(a for (a, b) in REPLACE_EFF if b == rp)
        if not subs: return {"ok": False, "log": log + [f"{rp} không sẵn sàng và không có luật Replace"]}
        use = subs[0]; log.append(f"{rp} không sẵn sàng → luật Replace: dùng {use}")
    try: ans = rp_answer(use, topic, school, mirror_age)
    except PermissionError as e: return {"ok": False, "log": log + [str(e)]}
    if ans["age_days"] > MAX_AGE_DAYS:
        return {"ok": False, "log": log + [f"RC constraint c3: dữ liệu {ans['age_days']} ngày > {MAX_AGE_DAYS} ngày → từ chối"]}
    return {"ok": True, "answer": ans, "log": log + [f"✔ {ans}"]}

def show_fed(title, r): print(f"\n### {title}"); [print("  ", l) for l in r["log"]]
show_fed("Scenario 1 — RM chuyển trực tiếp tới RP", federated_query("RC_TeacherPortal", "GP"))
show_fed("Scenario 3 — RM_1 không nối RP_MS, phải hỏi RM_2", federated_query("RC_TeacherPortal", "MS"))
show_fed("Replace — RP_MS ngừng hoạt động, dùng RP_Mirror (đồng bộ 3 ngày)", federated_query("RC_ParentApp", "MS", down={"RP_MS"}, mirror_age=3))
show_fed("c3 — mirror cũ 12 ngày bị RC từ chối", federated_query("RC_ParentApp", "MS", down={"RP_MS"}, mirror_age=12))
show_fed("p2 — yêu cầu dữ liệu 'personal' bị RP từ chối", federated_query("RC_ParentApp", "GP", topic="personal"))

# Scenario 2: RM tổng hợp kết quả từng phần từ nhiều RP
parts = [federated_query("RC_TeacherPortal", s) for s in SCHOOLS]
print("\n### Scenario 2 — RM hợp nhất kết quả từ nhiều RP")
print(pd.DataFrame([p["answer"] for p in parts if p["ok"]]))

## Tổng kết
| Ý chính của bài báo | Điều notebook cho thấy |
|---|---|
| Cây ≤ 5 tầng cho KV/WC/Doc; đồ thị cho graph | Bảng số nút theo tầng (Phần 2). KV dừng ở tầng 4 |
| Thuộc tính = nút trên đường đi; 1 nút là đủ; quyền được thừa hưởng | `spec_match` và các luật R1–R7 (Phần 3) |
| KV khó FGAC, phải dựng nhánh ngang | Số khoá tăng khi tách `sid#group` (Phần 3) |
| Graph: không giới hạn tầng, cạnh = hành động, RBAC bằng nút role | `PG` và `graph_decide`; đồng nhất với ABAC trên cây (Phần 4) |
| Coarse-grained là phổ biến | So sánh số trường bị lộ thừa (4.1) |
| Injection, consistency, performance, audit | 4.2, 4.4, 4.5, 4.6 |
| Điều kiện môi trường, NLP→rule, lai RDBMS | 4.7, 4.8, 4.9 |
| Federation RP/RM/RC, luật = tập quan hệ, lưu triple | Phần 6 |

**Hạn chế cần lưu ý**
- Đây là **mô phỏng trong bộ nhớ** để minh hoạ khái niệm. Tính năng ủy quyền của từng sản phẩm NoSQL thực tế khác nhau và không được kiểm chứng ở đây.
- Người dùng/vai trò và các luật là **dữ liệu giả lập** do mình đặt ra. Chỉ dữ liệu học sinh là dataset thật (UCI) khi tải được.
- Bước NLP ở 4.8 là regex đơn giản, không phải mô hình AI.